# 2. Water: Getting Started

What can we explore with computational chemistry? Let us **explore the electronic structure** of some small molecules. In this chapter we will learn ways to input molecules into *GAMESS* via input files and interpret the resulting log files. We will run every job and read every log file from this notebook with *Python*.

## Learning Goals

This exercise will hopefully accomplish the following goals while learning new skills:
-	We will confirm that your installation of *GAMESS* is **working properly**
-	We will **construct** a *GAMESS* input file with a text editor, and later let *Python* write new input files for us.
-	We will use three methods for creating a description of the **atomic positions**
    -	Cartesian coordinates (classic *xyz* 3D space)
    -	Unique Cartesian coordinates with symmetry point groups
    -	Internal coordinates using a *Z-matrix*
-	We will use *GAMESS* to determine the **optimized structure** of small molecules using semi-empirical methods.
-	We will create new input files **using the results** of the previous calculation and calculate the optimized structure using **more expensive** methods.
-	We will humbly recognize that we are organic chemists and that we are using *GAMESS* as a **recipe** with poor understanding of the **consequences** of our choices. We will observe these consequences in our results as we move forward. If we knew what we were doing, we might avoid many inevitable **pitfalls**. We have a whole box a band aids and always wear our safety goggles. Let’s go!

## Small Molecules

**Methane**, **ammonia** and **water**: these three molecules are *sp<sup>3</sup>* hybridized and should be **tetrahedral** in their VSEPR geometry. We expect to see bonds or lone pairs at angles of near 109&deg;

To investigate these molecules, will need to **describe** a molecule for a computer. We will discuss three methods: Cartesian coordinates, where every atom is placed in 3D space with *xyz* values; **unique Cartesian coordinates**, where only atoms that are symmetry unique are placed and the rest are automatically generated to satisfy the symmetry point group; and the ***Z-matrix***, where each atom is placed in space relative to the previous one in a carefully created list.

*GAMESS* works by positioning the nuclei and then creating a mathematical function for the **energy** of the electrons. By taking the **derivative** of that function, *GAMESS* can determine if the current position is on a **slope** and then change the positions by a small amount to follow that slope downhill in energy. A new function is calculated and the process repeats again and again.

In this exercise, we will be **manually** creating input files and running them. In the future, we will learn how to use *MacMolPlt* and *Avogadro* to quickly build molecules and **automatically** create input files. But lets us first get our hands dirty before we start using power tools to do the hard work for us. Wax on, wax off!

### Settings

Run this cell first. It loads the book's helper module, `gamess_tools.py`, which finds *GAMESS* using the location you set in `gamess_settings.py` on the [setup page](1_Setup.ipynb). All the jobs in this notebook write their `.log` files into the `runs/` folder next to the notebook.

In [1]:
# ===== SETTINGS =========================================================
# Your GAMESS location is set ONCE, in gamess_settings.py (next to this
# notebook). Edit that file, save it, and restart the kernel.
from pathlib import Path
import gamess_tools as gt

INPUTS = Path("inputs")     # input files that come with the book
RUNS = Path("runs")         # where log files and new input files are written
RUNS.mkdir(exist_ok=True)

HARTREE_TO_KJMOL = 2625.5   # 1 hartree = 2625.5 kJ/mol (see "Units" below)
# ========================================================================

## Water

Water has three atoms that have found an **arrangement** in space for the nuclei and electrons that is as **low in energy** as possible. Can we calculate that arrangement and see if it fits with observations? First, we must create a data set that describes the positions of the nuclei. Let us begin by discussing the nature of the 3D **coordinate space** used by *GAMESS*.

### The 3D World

*x*,*y*,*z*… That triplet describes the **entire universe**. But *GAMESS* does not treat *x*, *y* and *z* equally. Consider the axis diagram in {numref}`fig1`.

```{figure} images/GAMESS_AXES.png
---
width: 500px
name: fig1
---
The axis scheme for *GAMESS* atomic coordinates
```

When constructing a structure using either Cartesian coordinates or a *Z-matrix*, we don’t consider any priority for an axis. However, when using **point groups** and symmetry to define molecules with **unique** Cartesian coordinates, we must pay attention to how *GAMESS* manipulates **symmetry**. The **primary axis** of the point group is expected to be the *z*-axis. Any vertical &sigma;-plane should first be set up in the *xz* plane. The horizontal &sigma;-plane should be considered to be the *xy* plane. The first perpendicular 2-fold axis should be set up along the *x*-axis. Keep these ideas in mind as we construct our first molecules.

### Symmetry

Symmetry is beautiful. It also helps with math. If a molecule has symmetry, we should use it to make the calculations performed by *GAMESS* **less expensive**. A molecule that is known to possess 2-fold symmetry may be half as difficult to analyze because we may only need half the math. This depends a lot on the basis set chosen to describe each atomic orbital but, most of the time, it is to our advantage to declare and use a **symmetry point group**.

**Water** has a *C<sub>2</sub>* axis. There is no perpendicular *C<sub>2</sub>* axis so we remain in the *C* group rather than rising to *D*. There is no *&sigma;<sub>h</sub>* but there are two *&sigma;<sub>v</sub>* planes. Therefore water is in the *C<sub>2v</sub>* point group. We will use Cartesian coordinates that place the **primary axis** along the *z*-axis and the atoms in the *xz* plane. See {numref}`fig2` for how we will position water in the 3D space used by *GAMESS*.

```{figure} images/WATER_AXES.png
---
width: 500px
name: fig2
---
Water positioned in *GAMESS* 3D space with the *C<sub>2</sub>* axis along the *z*-axis and the primary *&sigma;<sub>v</sub>* plane as the *xz* plane
```

### Building Water

So how shall we build water? We can provide Cartesian coordinates that position **oxygen** at the origin. Then we can position a **hydrogen** in one *xz* quadrant and the second hydrogen in the opposite quadrant. Compare the below set of coordinates to {numref}`fig2`.

| Atom	   |   x   | y     | z     |
|:-----    | :---: | :---: | :---: |
| Oxygen   | 0	   | 0	   | 0     |
| Hydrogen | b	   | 0	   | –a    |
| Hydrogen | –b    | 0     | –a    |

What values should we use for *a* and *b*? We know that the **bond angle** should be 104.5&deg; and that the **bond length** in water is 0.95 Å. We could use trigonometry and position the atoms accordingly. Or we could just throw some **guess** in and let *GAMESS* start there as it moves toward finding the optimal structure. I will set both *a* and *b* to 0.7 Å. Our initial guess at water will have a bond angle of 90&deg; and a bond length of $\sqrt{0.7^2+0.7^2} = 0.99$ Å.

### Your First Input File

We now need to create a **text file** that will be read into *GAMESS* to provide the information for the calculation. We must tell *GAMESS* what **kind** of calculation, the **methods** to use for determining energy, the **conditions** for declaring a satisfactory structure, and the **data** for the structure.

Below in {numref}`water-inp` is the text file for this first *GAMESS* run. I created a file called `water.inp` in the `inputs/` folder and typed in the following text with a text editor (you can also create a new text file from the *Jupyter* file browser).

```{literalinclude} inputs/water.inp
:label: water-inp
:caption: inputs/water.inp
:linenos: True
```

### The Structure of The Input File

A *GAMESS* input file is a series of text lines called “**cards**.” (Before your parents were born, computers did not receive input via a keyboard, they used punched paper cards.) Each line must be less than **80 characters** (those paper cards were only so big.)

The input is separated in a series a “**groups**.” Each group begins with a `$` character and ends with a `$END` string. There are many, many groups available to the sophisticated theoretical chemist. We will only use a handful. In the `water.inp` file, we use the following groups: `$CONTRL`, `$SYSTEM`, `$BASIS`, `$SCF`, `$STATPT` and `$DATA`. The `$` character for a group MUST be in the **second column** of a card (*GAMESS* is written in *FORTRAN*, an ancient computer language that is based on paper cards and therefore obsessive about columns in data input.) After several *GAMESS* jobs fail spectacularly on you, you will come to believe in this second column requirement.

Here is a quick *Python* check of both rules for our file. It is not something *GAMESS* needs, but it is a handy habit when a job fails for no obvious reason.

In [2]:
for number, line in enumerate((INPUTS / "water.inp").read_text().splitlines(), start=1):
    problems = []
    if len(line) >= 80:
        problems.append(f"{len(line)} characters long")
    if "$" in line and not line.startswith(" $"):
        problems.append("$ is not in column 2")
    print(f"{number:3d} {line:<60} {'; '.join(problems) or 'ok'}")

  1  $CONTRL SCFTYP=RHF RUNTYP=OPTIMIZE MAXIT=30 MULT=1 ICHARG=0 COORD=CART $END ok
  2  $SYSTEM TIMLIM=525600 MEMORY=1000000 $END                   ok
  3  $BASIS GBASIS=AM1 $END                                      ok
  4  $SCF DIRSCF=.TRUE. $END                                     ok
  5  $STATPT OPTTOL=0.0001 NSTEP=100 $END                        ok
  6  $DATA                                                       ok
  7 Water                                                        ok
  8 CNV 2                                                        ok
  9                                                              ok
 10 O     8.0   0.0   0.0   0.0                                  ok
 11 H     1.0  -0.7   0.0  -0.7                                  ok
 12 H     1.0   0.7   0.0  -0.7                                  ok
 13  $END                                                        ok


### *GAMESS* groups

Here is a quick **overview** of the groups used in our first job. We will add to our skills as we move forward.

#### The \$CONTRL Group

The `$CONTRL` group establishes the **kind of calculation**. It may contain many parameters. The ones used here are our beginning set.

```{code-block} text
:caption: water.inp
:linenos: True
:lineno-start: 1

 $CONTRL SCFTYP=RHF RUNTYP=OPTIMIZE MAXIT=30 MULT=1 ICHARG=0 COORD=CART $END
```

| Parameter | Notes |
| :---- | :---- |
| `SCFTYP=RHF` | The **method** used is “restricted Hartree-Fock.” This assumes that all electrons are paired. |
| `RUNTYP=OPTIMIZE` | The **calculation** is a structural optimization. |
| `MAXIT=30` | The **maximum** number of iterations to determine the derivative of the function that describes everything about everything in the molecule. |
| `MULT=1` | The **spin** multiplicity (1 means all electron spins are paired) |
| `ICHARG=0` | The **charge** on the molecule |
| `COORD=CART` | The **coordinate** system that we are using. We will use `CART` for describing the *xyz* of all atoms, `UNIQUE` for describing the *xyz* coordinates of atoms unique in the symmetry point group, and `ZMT` for internal coordinates in *Z-matrix* format. The very first run with `COORD=CART` will convert the *xyz* data into a symmetrical system aligned with the **primary axis** (we are already there) and with the origin at the center of mass of the molecule (we are definitely not there). The first result file will report the unique symmetry coordinates for the molecule and we should use these new coordinates and `COORD=UNIQUE` for all subsequent calculations. |

The `CART` coordinate option has been deprecated in modern versions of *GAMESS*. The new flag is `PRINAXIS`. The `CART` option will perform the job of `PRINAXIS`, but `CART` may someday be gone entirely. Both `CART` and `PRINAXIS` will reorient the molecule along the primary axis of the point group and generate a set of Cartesian coordinates for the symmetry unique atoms. You will only ever run a `CART` or `PRINAXIS` calculation once and thereafter will use the symmetry unique atoms and `COORD=UNIQUE` in all future calculations with your molecule.

#### The \$SYSTEM Group

The `$SYSTEM` group defines parameters for the computer system.

```{code-block} text
:caption: water.inp
:linenos: True
:lineno-start: 2

 $SYSTEM TIMLIM=525600 MEMORY=1000000 $END
```

| Parameter | Notes |
| :---- | :---- |
|`TIMLIM=525600` | The **time limit** for the calculation in minutes. 525,600 minutes – how do you measure a year in the life? |
|`MEMORY=1000000` | The **memory** to be used (for each processor core), counted in 8-byte words. Increase this if your job crashes out and complains about a lack of memory. |

#### The \$BASIS Group

The `$BASIS` group defines the **theoretical rigor** of the calculation. We will discuss this in more detail later. By using the AM1 level of theory we are declaring that we will use **semi-empirical** methods. This is a very **low level** calculation and requires little in computer resources. We will soon explore methods that will take some serious computer time (and time is money.)

```{code-block} text
:caption: water.inp
:linenos: True
:lineno-start: 3

 $BASIS GBASIS=AM1 $END
```

#### The \$SCF Group

The `$SCF` group is specific to the way the calculation is **performed**. Many of the integrals needed to compute the energy can be calculated once, stored in a temporary file on disk and read back each time they are needed (they are needed many times in the calculation). Modern processors can actually **recalculate** them from scratch faster than a computer hard disk can spin. The `DIRSCF=.TRUE.` command sets *GAMESS* to recalculate them **each time** they are needed rather than read from the much slower disk drive (modern SSD storage may change this requirement – computers are improving all the time.)

```{code-block} text
:caption: water.inp
:linenos: True
:lineno-start: 4

 $SCF DIRSCF=.TRUE. $END
```

#### The \$STATPT Group

The `$STATPT` group establishes the parameters for deciding when we have reached a “**stationary point**” where the derivative of the energy function is near zero (zero slope is likely the bottom of an energy well, the lowest point along the path that *GAMESS* has taken).

```{code-block} text
:caption: water.inp
:linenos: True
:lineno-start: 5

 $STATPT OPTTOL=0.0001 NSTEP=100 $END
```

| Parameter | Notes |
| :---- | :---- |
|`OPTTOL=0.0001` | If the largest energy gradient (slope) in the system is less than this value (and the RMS average gradient is below a third of it) then we declare that we are **finished**. |
|`NSTEP=100` | The maximum number of steps that we take in the algorithm before we **give up**. |

#### The \$DATA Group

The above groups instructed *GAMESS* how to treat the next group, the **star of the show**, the `$DATA` group. Let us now examine this important part of the input file.

```{code-block} text
:caption: water.inp
:linenos: True
:lineno-start: 6
:emphasize-lines: 8

 $DATA
Water
CNV 2

O     8.0   0.0   0.0   0.0
H     1.0  -0.7   0.0  -0.7
H     1.0   0.7   0.0  -0.7
 $END
```

| Line | Notes |
| :---- | :---- |
| Line 7 | The first line in the `$DATA` group is the **name** of the job. Put whatever you want there. I chose water, it seemed appropriate. |
| Line 8 | The next line is the **symmetry point group**. Water is *C<sub>2v</sub>*. *GAMESS* establishes this in two parts: first we declare *C<sub>nv</sub>* and then we set the value of $n$ as 2.|
| Line 9 | The next line is **blank**. This is important. Leave it blank. (If we are not using symmetry, we will declare the point group to be *C<sub>1</sub>* and then the blank line will not be used.  Don’t worry, after a few crashed jobs you will get the hang of this.) |
| Lines 10..n | All the following lines are the data for the nuclei. The first column is the **name** (it can be **anything** up to 10 characters), the next is the **nuclear charge** (this **defines** the atom), and then the *xyz* coordinates. Like all *GAMESS* groups, the `$DATA` group is ended by the `$END` string. |

Since we defined `COORD=CART` in the `$CONTRL` group, *GAMESS* will input all the **atomic coordinates** and attempt to interpret them via the *C<sub>2v</sub>* point group. If your coordinates do not fit in that point group, the calculation will probably fail. If they fit the *C<sub>2v</sub>* group, then *GAMESS* will identify the symmetry and use only the **unique atoms** within the point group in the subsequent calculation. Defining the symmetry point group is very useful in **minimizing the costs** of a calculation.

-----

## Your First *GAMESS* Job

Now we have an **input file** named `water.inp`. We run it with one line of *Python*. The function `run_gamess()` calls the *rungms* script, writes the result to `runs/water.log` and tells us whether *GAMESS* finished normally. In a few seconds the job is finished. That’s all there is to it.

In [3]:
water_log = gt.run_gamess(INPUTS / "water.inp")

water: finished normally (water.log)


```{tip}
If the job fails at once, check the location of *GAMESS* in `gamess_settings.py` and run `gt.check_installation()` as on the [setup page](1_Setup.ipynb). On a Mac, the shared-memory limit is reset each time the computer restarts.
```

-----

## Your First Output File

The result file is named `runs/water.log` in this case. You can open it in a **text editor** and examine the contents, but it is several hundred lines long. Let *Python* do the searching.

**Search** for the string `NSERCH`. Every time it appears involves setting up or declaring the results of an energy calculation. The helper `gt.grep()` returns every line that contains a piece of text.

In [4]:
for line in gt.grep("NSERCH", water_log):
    print(line)

 BEGINNING GEOMETRY SEARCH POINT NSERCH=   0 ...
          NSERCH=  0     ENERGY=     -12.8027679
 NSERCH:   0  E=      -12.8027678782  GRAD. MAX=  0.0593284  R.M.S.=  0.0244052
 BEGINNING GEOMETRY SEARCH POINT NSERCH=   1 ...
          NSERCH=  1     ENERGY=     -12.8083173
 NSERCH:   1  E=      -12.8083172648  GRAD. MAX=  0.0246683  R.M.S.=  0.0100711
 BEGINNING GEOMETRY SEARCH POINT NSERCH=   2 ...
          NSERCH=  2     ENERGY=     -12.8093112
 NSERCH:   2  E=      -12.8093111534  GRAD. MAX=  0.0007158  R.M.S.=  0.0002944
 BEGINNING GEOMETRY SEARCH POINT NSERCH=   3 ...
          NSERCH=  3     ENERGY=     -12.8093121
 NSERCH:   3  E=      -12.8093121210  GRAD. MAX=  0.0000133  R.M.S.=  0.0000072


The first time it appears we are at the **beginning** of the geometry search. The function `gt.excerpt()` prints the lines that follow a piece of text, with their **line numbers** in the log file. We ask for the first occurrence (`occurrence=0`).

In [5]:
gt.excerpt(water_log, "BEGINNING GEOMETRY SEARCH POINT NSERCH=   0", after=12, occurrence=0)

  465   BEGINNING GEOMETRY SEARCH POINT NSERCH=   0 ...
  466  
  467   COORDINATES OF SYMMETRY UNIQUE ATOMS (ANGS)
  468     ATOM   CHARGE       X              Y              Z
  469   ------------------------------------------------------------
  470   O           8.0   0.0000000000   0.0000000000   0.0783404292
  471   H           1.0  -0.7000000000   0.0000000000  -0.6216595708
  472   COORDINATES OF ALL ATOMS ARE (ANGS)
  473     ATOM   CHARGE       X              Y              Z
  474   ------------------------------------------------------------
  475   O           8.0   0.0000000000   0.0000000000   0.0783404292
  476   H           1.0   0.7000000000  -0.0000000000  -0.6216595708
  477   H           1.0  -0.7000000000   0.0000000000  -0.6216595708


We see that *GAMESS* has found the **unique atoms** of the **point group**. From these, all the atoms can be created. *GAMESS* is using our initial values but has positioned the center of mass at the origin by adding 0.0783 Å to the *z*-axis.

Keep searching for `NSERCH`. Almost **150 lines later** in the `water.log` result file we see that we have finished step zero.

In [6]:
gt.excerpt(water_log, "NSERCH:   0", after=21, occurrence=0)

  612   NSERCH:   0  E=      -12.8027678782  GRAD. MAX=  0.0593284  R.M.S.=  0.0244052
  613  
  614            FORCE CONSTANT MATRIX NOT UPDATED --- TAKING FIRST STEP
  615            MIN SEARCH, CORRECT HESSIAN, TRYING PURE NR STEP
  616                 NR STEP HAS LENGTH         =   0.474433
  617            TRIM/QA LAMBDA FOR NON-TS MODES =  -0.06454854
  618            TRIM/QA STEP HAS LENGTH         =   0.300000
  619            RADIUS OF STEP TAKEN=   0.30000  CURRENT TRUST RADIUS=   0.30000
  620  
  621   BEGINNING GEOMETRY SEARCH POINT NSERCH=   1 ...
  622  
  623   COORDINATES OF SYMMETRY UNIQUE ATOMS (ANGS)
  624     ATOM   CHARGE       X              Y              Z
  625   ------------------------------------------------------------
  626   O           8.0   0.0000000000   0.0000000000  -0.0197579410
  627   H           1.0  -0.7733745023  -0.0000000000  -0.5726103857
  628   COORDINATES OF ALL ATOMS ARE (ANGS)
  629     ATOM   CHARGE       X              Y             

The initial **energy** and the derivative (slope or gradient of the energy function) has been **calculated**. The gradient w.r.t. each axis will be used to compute a distance to **move** for each atom. Each change is **applied** and the data set for step one is declared a few lines later.

The energy and gradients will be computed again and the coordinates altered accordingly. This will **repeat** until we fulfil the conditions for the stationary point set in the `$STATPT` group. In our example, the **stationary point** is declared to be located after we finish step 3. We show the **last** `NSERCH:` line and what follows it.

In [7]:
gt.excerpt(water_log, "NSERCH:", after=26)

  875   NSERCH:   3  E=      -12.8093121210  GRAD. MAX=  0.0000133  R.M.S.=  0.0000072
  876  
  877  
  878        ***** EQUILIBRIUM GEOMETRY LOCATED *****
  879   COORDINATES OF SYMMETRY UNIQUE ATOMS (ANGS)
  880     ATOM   CHARGE       X              Y              Z
  881   ------------------------------------------------------------
  882   O           8.0   0.0000000000   0.0000000000   0.0083001974
  883   H           1.0  -0.7550350900  -0.0000000000  -0.5866394549
  884   COORDINATES OF ALL ATOMS ARE (ANGS)
  885     ATOM   CHARGE       X              Y              Z
  886   ------------------------------------------------------------
  887   O           8.0   0.0000000000   0.0000000000   0.0083001974
  888   H           1.0   0.7550350900  -0.0000000000  -0.5866394549
  889   H           1.0  -0.7550350900  -0.0000000000  -0.5866394549
  890  
  891            INTERNUCLEAR DISTANCES (ANGS.)
  892            ------------------------------
  893  
  894                  1 O  

We now have the **energy** that was calculated and the coordinates for the final iteration of the **structure**. Next, *GAMESS* will write molecular orbital data and any other sets of data requested in the input file. We will use this extra data to visualize molecular orbitals in a future exercise.

We see that the **bond lengths** are 0.961 Å and are identical, as required by the symmetry point group.

### Letting *Python* read the numbers

Instead of reading numbers off the screen we can ask *Python* for them. `gt.nserch_energies()` lists the energy at each step and `gt.final_geometry()` returns the last set of coordinates as a list of `(name, charge, x, y, z)`. With the geometry in hand, `gt.distance()` and `gt.angle()` do the trigonometry for us (atoms are numbered from 1, in the order *GAMESS* printed them).

In [8]:
for step, energy in gt.nserch_energies(water_log):
    print(f"step {step}:  E = {energy:.10f} hartree")

water_geom = gt.final_geometry(water_log)
print(f"\nO-H bond length: {gt.distance(water_geom, 1, 2):.4f} Å")
print(f"H-O-H angle:     {gt.angle(water_geom, 2, 1, 3):.2f}°")

step 0:  E = -12.8027678782 hartree
step 1:  E = -12.8083172648 hartree
step 2:  E = -12.8093111534 hartree
step 3:  E = -12.8093121210 hartree

O-H bond length: 0.9613 Å
H-O-H angle:     103.53°


## Higher Levels of Theory

AM1 uses precalculated values and assumptions to **simplify** the math. It is good for systems that it has been designed for, mostly simple organic molecules. However, if we want to explore strained systems or calculate **accurate** molecular orbitals we will have to use *ab initio* methods. We will use Hartree-Fock theory and basis sets of atomic orbitals to get more accurate answers.

```{note}
Don't think about it too much. The two levels of rigour described below are nowhere near the state of the art, but they will serve for what we need to do. To understand the theory and the choice of orbital basis sets you should enjoy the **computational chemistry course** that is taking place in parallel with this course. We will use *GAMESS* without having a clue what we are doing. Lives are not at stake here.
```

### A Higher Level: HF/3-21G

It took my little laptop computer 0.2 seconds to calculate the final structure and energy of water at the AM1 level. AM1 uses pre-configured functions to enable fast calculations but may not be accurate in all situations (although small neutral molecules should give **excellent results**.)

The **next level** up in mathematical rigour is Hartree-Fock. HF theory attempts to describe the atomic orbitals using mathematical **approximations** but makes no simplifications. It does ignore electron correlation and the Restricted Hartree Fock method (RHF) that we will be using enforces that all electrons are **paired** in molecular orbitals. The math approximations that describe the atomic orbitals use Gaussian functions to model the atomic orbitals. **Gaussian functions** are far easier for a computer to calculate and adding many Gaussians together can get very close to the real answer. The more Gaussians and the more ways they are combined will give better accuracy but will also be more expensive in computer time.

We will use 3-21G, a very low cost basis set and 6-311G(d,p)++, a much more complete and much more expensive basis set in all our work.

We will change the `$BASIS` group in the `water.inp` file to use the 3-21G basis set

```text
 $BASIS GBASIS=N21 NGAUSS=3 $END
```

We will also copy the final coordinates from the AM1 job into the new input file. We should always use the work of those who have gone before to advance our goals. We stand on the shoulders of giants. When we work up to very **expensive** methods, it is best to start with an optimized structure from a **cheaper** method.

With a text editor we would copy `water.inp`, edit three lines and **cut and paste** the coordinates of the symmetry unique atoms from `water.log`. In *Python*, `gt.final_geometry(log, unique=True)` gives us those coordinates and `gt.new_input()` writes a copy of an input file with new atom lines and with any text we name swapped for new text.

In [9]:
am1_unique = gt.final_geometry(water_log, unique=True)
print(gt.atom_lines(am1_unique))

 O        8.0    0.0000000000    0.0000000000    0.0083001974
 H        1.0   -0.7550350900   -0.0000000000   -0.5866394549


In [10]:
water_321G_inp = gt.new_input(
    INPUTS / "water.inp", RUNS / "water_321G.inp",
    atoms=am1_unique,
    replacements={
        "COORD=CART": "COORD=UNIQUE",
        "GBASIS=AM1": "GBASIS=N21 NGAUSS=3",
        "NSTEP=100": "NSTEP=20",
        "Water\n": "Water 3-21G\n",
    },
)
print(water_321G_inp.read_text())

 $CONTRL SCFTYP=RHF RUNTYP=OPTIMIZE MAXIT=30 MULT=1 ICHARG=0 COORD=UNIQUE $END
 $SYSTEM TIMLIM=525600 MEMORY=1000000 $END
 $BASIS GBASIS=N21 NGAUSS=3 $END
 $SCF DIRSCF=.TRUE. $END
 $STATPT OPTTOL=0.0001 NSTEP=20 $END
 $DATA 
Water 3-21G
CNV 2

 O        8.0    0.0000000000    0.0000000000    0.0083001974
 H        1.0   -0.7550350900   -0.0000000000   -0.5866394549
 $END



The **changes** from the original input file are the basis set on line 3, the job title, and the coordinates. Note that the `COORD` parameter has been changed to `UNIQUE`. *GAMESS* reported the *xyz* coordinates of the **symmetry unique atoms** in the previous output file. Since we are now using unique coordinates we must inform the software of that fact.

Now we run the job.

In [11]:
water_321G_log = gt.run_gamess(water_321G_inp)
gt.excerpt(water_321G_log, "NSERCH:", after=24)

water_321G: finished normally (water_321G.log)
 1209   NSERCH:   5  E=      -75.5859597578  GRAD. MAX=  0.0000143  R.M.S.=  0.0000060
 1210  
 1211  
 1212        ***** EQUILIBRIUM GEOMETRY LOCATED *****
 1213   COORDINATES OF SYMMETRY UNIQUE ATOMS (ANGS)
 1214     ATOM   CHARGE       X              Y              Z
 1215   ------------------------------------------------------------
 1216   O           8.0   0.0000000000   0.0000000000  -0.0081020518
 1217   H           1.0  -0.7804697613   0.0000000000  -0.5784383303
 1218   COORDINATES OF ALL ATOMS ARE (ANGS)
 1219     ATOM   CHARGE       X              Y              Z
 1220   ------------------------------------------------------------
 1221   O           8.0   0.0000000000   0.0000000000  -0.0081020518
 1222   H           1.0   0.7804697613  -0.0000000000  -0.5784383303
 1223   H           1.0  -0.7804697613   0.0000000000  -0.5784383303
 1224  
 1225            INTERNUCLEAR DISTANCES (ANGS.)
 1226            --------------------

A **result file** named `water_321G.log` was created. We looked for the last occurrence of `NSERCH` and found the results. The job reached a stationary point after five optimization steps. We now have the energy and structure at the HF/3-21G level.

The **bond lengths** are slightly longer and the calculated **energy** is very different. We **can’t compare** AM1 energy values to HF because AM1 is a very different method. However we **can compare** HF/3-21G to HF/6-311G(d,p)++. It is the same method, but with a more complete basis set.

### More Power: HF/6-311G(d,p)++

By changing the `$BASIS` group to the following we will be implementing the 6-311G(d,p)++ basis set.

```text
 $BASIS GBASIS=N311 NGAUSS=6 NDFUNC=1 NPFUNC=1 DIFFSP=.TRUE. DIFFS=.TRUE. $END
```

We will use the coordinate data from the 3-21G run to start as **close** to the true answer **as possible**. This time we start from the 3-21G input file, which already has `COORD=UNIQUE`.

In [12]:
BASIS_6311 = "GBASIS=N311 NGAUSS=6 NDFUNC=1 NPFUNC=1 DIFFSP=.TRUE. DIFFS=.TRUE."

water_6311G_inp = gt.new_input(
    water_321G_inp, RUNS / "water_6311G.inp",
    atoms=gt.final_geometry(water_321G_log, unique=True),
    replacements={"GBASIS=N21 NGAUSS=3": BASIS_6311,
                  "Water 3-21G": "Water 6-311G(d,p)++"},
)
print(water_6311G_inp.read_text())

water_6311G_log = gt.run_gamess(water_6311G_inp)
gt.excerpt(water_6311G_log, "NSERCH:", after=24)

 $CONTRL SCFTYP=RHF RUNTYP=OPTIMIZE MAXIT=30 MULT=1 ICHARG=0 COORD=UNIQUE $END
 $SYSTEM TIMLIM=525600 MEMORY=1000000 $END
 $BASIS GBASIS=N311 NGAUSS=6 NDFUNC=1 NPFUNC=1 DIFFSP=.TRUE. DIFFS=.TRUE. $END
 $SCF DIRSCF=.TRUE. $END
 $STATPT OPTTOL=0.0001 NSTEP=20 $END
 $DATA 
Water 6-311G(d,p)++
CNV 2

 O        8.0    0.0000000000    0.0000000000   -0.0081020518
 H        1.0   -0.7804697613    0.0000000000   -0.5784383303
 $END



water_6311G: finished normally (water_6311G.log)
 1535   NSERCH:   5  E=      -76.0534461703  GRAD. MAX=  0.0000111  R.M.S.=  0.0000051
 1536  
 1537  
 1538        ***** EQUILIBRIUM GEOMETRY LOCATED *****
 1539   COORDINATES OF SYMMETRY UNIQUE ATOMS (ANGS)
 1540     ATOM   CHARGE       X              Y              Z
 1541   ------------------------------------------------------------
 1542   O           8.0   0.0000000000   0.0000000000  -0.0116217188
 1543   H           1.0  -0.7527830929  -0.0000000000  -0.5766784968
 1544   COORDINATES OF ALL ATOMS ARE (ANGS)
 1545     ATOM   CHARGE       X              Y              Z
 1546   ------------------------------------------------------------
 1547   O           8.0   0.0000000000   0.0000000000  -0.0116217188
 1548   H           1.0   0.7527830929  -0.0000000000  -0.5766784968
 1549   H           1.0  -0.7527830929  -0.0000000000  -0.5766784968
 1550  
 1551            INTERNUCLEAR DISTANCES (ANGS.)
 1552            ------------------

Let us put the two Hartree-Fock results side by side.

In [13]:
for label, log in [("HF/3-21G", water_321G_log), ("HF/6-311G(d,p)++", water_6311G_log)]:
    geom = gt.final_geometry(log)
    energy = gt.nserch_energies(log)[-1][1]
    print(f"{label:18s} E = {energy:.10f} hartree   "
          f"O-H = {gt.distance(geom, 1, 2):.4f} Å   H-O-H = {gt.angle(geom, 2, 1, 3):.2f}°")

HF/3-21G           E = -75.5859597578 hartree   O-H = 0.9667 Å   H-O-H = 107.68°
HF/6-311G(d,p)++   E = -76.0534461703 hartree   O-H = 0.9413 Å   H-O-H = 106.21°


The HF/6-311G(d,p)++ energy was –76.0534 hartree whereas the HF/3-21G energy was –75.5860 hartree. **Higher** levels of theory will give **lower** energy values.  The bond length is also now shorter; is that closer to the truth?

```{note}
In general, a calculated energy is always higher than the “true” energy. Only a perfect basis set that includes an exact model for the atomic orbitals will give the lowest value possible. Nothing is perfect, but we can get close if we take the time to run more expensive methods and basis sets.
```

What if we **calculate** the HF/6-311G(d,p)++ energy **using** the HF/3-21G optimized structure? This is a *single point* energy calculation: we change `RUNTYP=OPTIMIZE` to `RUNTYP=ENERGY` and *GAMESS* computes the energy once, without moving the atoms.

In [14]:
single_point_inp = gt.new_input(
    water_6311G_inp, RUNS / "water_6311G_E_321G.inp",
    atoms=gt.final_geometry(water_321G_log, unique=True),
    replacements={"RUNTYP=OPTIMIZE": "RUNTYP=ENERGY"},
)
single_point_log = gt.run_gamess(single_point_inp)

e_single = gt.final_energy(single_point_log)
e_opt = gt.nserch_energies(water_6311G_log)[-1][1]
print(f"HF/6-311G(d,p)++ energy at the HF/3-21G structure:  {e_single:.10f} hartree")
print(f"HF/6-311G(d,p)++ energy at its own optimum:         {e_opt:.10f} hartree")
print(f"difference: {(e_single - e_opt) * HARTREE_TO_KJMOL:.2f} kJ/mol")

water_6311G_E_321G: finished normally (water_6311G_E_321G.log)
HF/6-311G(d,p)++ energy at the HF/3-21G structure:  -76.0519616441 hartree
HF/6-311G(d,p)++ energy at its own optimum:         -76.0534461703 hartree
difference: 3.90 kJ/mol


We get an energy value of –76.0520 hartree. The HF/3-21G structure has an HF/6-311G(d,p)++ energy that is very close to the energy of the HF/6-311G(d,p)++ optimized structure. Perhaps we can use **cheaper** methods to calculate optimized structures and then **more expensive** methods for single point energy calculations. Quite often, stationary points are very similar between different levels of theory because we are dealing with relative energies and the bottom of the energy well is in the almost the same location, even if the absolute energy is not accurate. With more complex systems, such as transition states, we may need to optimize structure at higher levels of theory for accuracy.

## Units

Hartrees? You may notice that data tables in the output file report energy in **hartree units**, distances in both **angstroms** and **bohr atomic units** and angles in degrees and in radians. Deep inside *GAMESS*, the math uses bohr for distance and radians for angles in its calculations. It also uses the elementary charge and mass of the electron and Planck’s constant. The hartree is an energy unit derived from the **fundamental** measurements of a hydrogen atom and the intrinsic properties of space and time.

**Keep** this conversion factor close by: 1 hartree = 2,625.5 kJ/mol. (It is stored as `HARTREE_TO_KJMOL` in the settings cell at the top of this notebook.)

*GAMESS* will output distances and angles in **familiar** units such as angstroms and degrees but will always report energy in hartrees. One hartree is 4.3597 &times; 10<sup>−18</sup> J. When we multiply by Avogadro’s number we get 2625.50 kJ/mol.

In [15]:
hartree_J = 4.3597447222e-18     # J
avogadro = 6.02214076e23         # 1/mol
print(f"1 hartree = {hartree_J * avogadro / 1000:.2f} kJ/mol")

1 hartree = 2625.50 kJ/mol


## Internal Coordinates

For many of our exercises, we will be building molecules using **internal coordinates**. These coordinates define atomic positions relative the first atom defined in the list. This list is called the *Z-matrix*. A ***Z-matrix*** is a defined format for creating these internal coordinates.

Imagine three towns in a flat prairie, the last three towns on Earth. If I start at town A and travel 10 miles to town B, turn 45&deg; left and travel 5 miles to town C, then I can place the towns exactly, relative to each other. I will know the distance between A and C and all other angles from those first three coordinates.

Here is a *Z-matrix* for our town map…

```text
A
B  1  10
C  2   5  1  45
```
It can be interpreted as…

| Line | Notes |
| :---- | :---- |
|  1 |	Town A is declared to **exist** |
|  2 |	Town B is connected to line 1 by a **distance** of 10 miles |
|  3 |	Town C is connected to line 2 by a **distance** of 5 miles \& the **angle** through line 2 to line 1 is 45&deg;|

### The *Z-matrix* with Water

Here is a *Z-matrix* for the molecule water. It is interpreted in {numref}`fig3`

```text
Water
CNV 2

H
O  1  1.0
H  2  1.0  1  109.5
```

```{figure} images/Zmatrix.png
---
width: 500px
name: fig3
---
Parsing the *Z-matrix* for water
```

*GAMESS* will **attempt** to fit the molecule to your declared symmetry point group and define a principal axis, etc. However, if your initial structure does not fit the point group the calculation will **fail**. To fit the *C<sub>2v</sub>* point group, the two bond lengths must be the same in the *Z-matrix* for water. One way that we can make sure that this is true is to use **variables** in the *Z-matrix*. Consider the example below.

```text
Water
CNV 2

H
O  1  bond_length
H  2  bond_length  1  bond_angle

bond_length=1.0
bond_angle=109.5
```

By using variables appropriately we can **make sure** that certain values are always identical, as required in a given point group. However, sometime we must be creative to enforce symmetry in a *Z-matrix*. {numref}`water_z_am1-inp1` shows the complete input file for an AM1 optimization of water using the *Z-matrix*.

```{literalinclude} inputs/water_Z_AM1.inp
:label: water_z_am1-inp1
:caption: inputs/water_Z_AM1.inp
:linenos: True
:emphasize-lines: 1
```

We see the *Z-matrix* used for the atomic coordinates in the `$DATA` group and we also see that the `COORD=CART` has been changed to `COORD=ZMT`.

When we submit the job, *GAMESS* will use the *Z-matrix* to create a set of Cartesian coordinates for the actual calculation. It will then attempt to identify the **primary axis** and apply the symmetry point group to create a set of **unique Cartesian coordinates**. It will then optimize the coordinates and, when a stationary point is reached, it will output the *xyz* coordinates, the unique coordinates and the new *Z-matrix*.

In [16]:
water_Z_log = gt.run_gamess(INPUTS / "water_Z_AM1.inp")
gt.excerpt(water_Z_log, "NSERCH:", after=19)

water_Z_AM1: finished normally (water_Z_AM1.log)
 1168   NSERCH:   6  E=      -12.8093121220  GRAD. MAX=  0.0000047  R.M.S.=  0.0000020
 1169  
 1170  
 1171        ***** EQUILIBRIUM GEOMETRY LOCATED *****
 1172   COORDINATES OF SYMMETRY UNIQUE ATOMS (ANGS)
 1173     ATOM   CHARGE       X              Y              Z
 1174   ------------------------------------------------------------
 1175   H           1.0  -0.7550622398   0.0000000000   0.5184764054
 1176   O           8.0  -0.0000000000   0.0000000000  -0.0764358674
 1177   COORDINATES OF ALL ATOMS ARE (ANGS)
 1178     ATOM   CHARGE       X              Y              Z
 1179   ------------------------------------------------------------
 1180   H           1.0   0.7550622398  -0.0000000000   0.5184764054
 1181   H           1.0  -0.7550622398   0.0000000000   0.5184764054
 1182   O           8.0  -0.0000000000   0.0000000000  -0.0764358674
 1183  
 1184   THE CURRENT FULLY SUBSTITUTED Z-MATRIX IS
 1185   H   
 1186   O      1   0

The *Z-matrix* is useful because we can **see** directly the **bond lengths and angles** in the result. We will not have to use trigonometry to calculate the bond angle from the *xyz* data. In the excerpt above we see that we reached an **optimized** geometry after 6 steps and that the AM1 bond lengths were calculated to be 0.961 Å and the bond angle to be 103.5&deg;. Note that *GAMESS* oriented the water molecule in the *xz* plane, as expected, but that it has the molecule **upside-down** compared to our original Cartesian input file described above. Interesting.

We could now copy the new *Z-matrix* or the unique coordinates into the HF/3-21G and the HF/6-311G(d,p)++ input files and **carry on** optimizing the structure at **higher** levels of theory. We should get identical structures and energies as the equivalent calculations that started with *xyz* coordinates because the molecule is the **same** no matter which coordinate system we use. Indeed, the AM1 energies of the two water jobs agree:

In [17]:
e_cart = gt.nserch_energies(water_log)[-1][1]
e_zmat = gt.nserch_energies(water_Z_log)[-1][1]
print(f"Cartesian start:  {e_cart:.10f} hartree")
print(f"Z-matrix start:   {e_zmat:.10f} hartree")
print(f"difference:       {(e_zmat - e_cart) * HARTREE_TO_KJMOL:.1e} kJ/mol")

Cartesian start:  -12.8093121210 hartree
Z-matrix start:   -12.8093121220 hartree
difference:       -2.6e-06 kJ/mol


### The *Z-matrix* for Ammonia

Water is a planar system (for the atoms) but **ammonia** is going to require a **third dimension**. Here is one option for the *Z-matrix*.

```{literalinclude} inputs/ammonia_Z.inp
:caption: inputs/ammonia_Z.inp
:lines: 6-14
```

This describes the *Z-matrix* as shown parsed in {numref}`fig4`

```{figure} images/Zmatrix_Ammonia.png
---
width: 500px
name: fig4
---
A *Z-matrix* for ammonia
```

Let us submit the job and see what happens.

In [18]:
ammonia_Z_log = gt.run_gamess(INPUTS / "ammonia_Z.inp")

ammonia_Z: FAILED, see /Users/blink/github/UsingGAMESSv2/runs/ammonia_Z.log


The **calculation failed**. Near the end of the short result file that was produced is an error message. We can find it by searching for `ERROR`.

In [19]:
gt.excerpt(ammonia_Z_log, "ERROR", after=5, occurrence=0)

  328   ERROR!
  329   YOUR CART/ZMT/ZMTMPC INPUT GENERATED   12 ATOMS,
  330   BUT ONLY    4 ATOMS WERE PRESENT IN YOUR $DATA.
  331   THIS MEANS THERE IS A MISTAKE IN YOUR COORDINATES,
  332   OR YOUR CHOICE OF GROUP.   ADIOS, MY FRIEND!!
  333  


Although the *Z-matrix* will create a structure that looks a lot like ammonia, the three **bond angles** between H and N will not all be identical in the *Z-matrix*. We defined the 3,1,2 and the 4,1,2 angles as 109.5&deg;, However we did not define the 4,1,3 bond angle. It will be the result of **positioning** the atoms according to the *Z-matrix* (1.0 Å from the nitrogen, 109.5&deg; from the 2,1 bond and 120&deg; from the 1,2,3 plane.) I thought that this would give us the correct structure. However, after the math is done that third 4,1,3 bond angle is **not exactly the same** as the stated 4,1,2 and 3,1,2 angles. As a result, *GAMESS* could not determine the correct primary axis because the structure was not an exact fit to *C<sub>3v</sub>* symmetry.

There is another way.

### Dummy Atoms in the *Z-matrix*

When constructing symmetrical molecules it can be useful to use **dummy atoms**. These are defined by the letter X in the *Z-matrix*.

```{literalinclude} inputs/ammonia_ZX.inp
:label: ammonia_zx-inp1
:caption: inputs/ammonia_ZX.inp
:linenos: True
```

The *Z-matrix* is outlined in {numref}`fig5` and the input file is shown in {numref}`ammonia_zx-inp1`.

```{figure} images/Dummy_Atoms.png
---
width: 500px
name: fig5
---
The *Z-matrix* for ammonia with a dummy atom to define symmetry
```

In [20]:
ammonia_ZX_log = gt.run_gamess(INPUTS / "ammonia_ZX.inp")
gt.excerpt(ammonia_ZX_log, "NSERCH:", after=26)

ammonia_ZX: finished normally (ammonia_ZX.log)
  971   NSERCH:   4  E=       -9.1354556208  GRAD. MAX=  0.0000006  R.M.S.=  0.0000004
  972  
  973  
  974        ***** EQUILIBRIUM GEOMETRY LOCATED *****
  975   COORDINATES OF SYMMETRY UNIQUE ATOMS (ANGS)
  976     ATOM   CHARGE       X              Y              Z
  977   ------------------------------------------------------------
  978   N           7.0   0.0000000000  -0.0000000000  -0.0630644261
  979   H           1.0   0.9385359335  -0.0000000000   0.2757943561
  980   COORDINATES OF ALL ATOMS ARE (ANGS)
  981     ATOM   CHARGE       X              Y              Z
  982   ------------------------------------------------------------
  983   N           7.0   0.0000000000  -0.0000000000  -0.0630644261
  984   H           1.0  -0.4692679667   0.8127959607   0.2757943561
  985   H           1.0  -0.4692679667  -0.8127959607   0.2757943561
  986   H           1.0   0.9385359335  -0.0000000000   0.2757943561
  987  
  988           

Using this matrix gives a **successful** calculation for the optimized geometry of ammonia at the AM1 level. In the excerpt above you will see that the *Z-matrix* was **not recreated**. This is because the **dummy atom** was used only in creating the initial *xyz* coordinates. It is not present in the calculation and *GAMESS* does not include it. *GAMESS* **cannot create** a *Z-matrix* that is guaranteed to fit the *C<sub>3v</sub>* symmetry without using dummy atoms, so it is sticking with Cartesian coordinates.

We can see that the AM1 bond **distances** were optimized at 0.998 Å. But we cannot easily see the H–N–H bond **angles** in the log file. *Python* can work them out from the coordinates:

In [21]:
nh3 = gt.final_geometry(ammonia_ZX_log)
print(f"N-H: {gt.distance(nh3, 1, 2):.4f} {gt.distance(nh3, 1, 3):.4f} {gt.distance(nh3, 1, 4):.4f} Å")
print(f"H-N-H: {gt.angle(nh3, 2, 1, 3):.2f} {gt.angle(nh3, 2, 1, 4):.2f} {gt.angle(nh3, 3, 1, 4):.2f}°")

N-H: 0.9978 0.9978 0.9978 Å
H-N-H: 109.09 109.09 109.09°


*GAMESS* can also do this for us, as we see next.

### The \$ZMAT Group

The above job essentially just **converted** the *Z-matrix* with a dummy atom to unique Cartesian coordinates for us. We can now use these Cartesian coordinates and ask *GAMESS* to **keep track** of the internal coordinates. We will create a new input file using the unique coordinates that were just generated by *GAMESS*.

We will perform the AM1 calculation **again** now with internal coordinates **declared** via a `$ZMAT` group. The `$ZMAT` group can be used to establish information about internal coordinates. Even if we submit the structure via Cartesian coordinates we can still have *GAMESS* use the internal coordinates of **our choice**. We no longer have a dummy atom, just the four atoms in NH<sub>3</sub>. Recall the original *Z-matrix* for ammonia that we built above. We can use those coordinates for our current calculation.

We must **declare** that we are using internal coordinates in the `$CONTRL` group by specifying **how many** will be used. We will add `NZVAR=6` to that group. There is a coordinate for each **degree of freedom** in a molecule. That is 3*n*–6 (3*n*–5 for linear molecules). For the four atoms of ammonia that is 3(4)–6 = 6 coordinates.

We will then construct the `$ZMAT` group as follows…

```text
 $ZMAT IZMAT(1)=1,2,1,
                1,3,1,  2,3,1,2,
                1,4,1,  2,4,1,2,  3,4,1,2,3   $END
```

The `IZMAT` matrix is populated by a series of numbers. Each **group** starts with a number and then is followed by the expected atom numbers. Here is how the `IZMAT` coordinates are declared.

- 1 indicates a **bond length** and is followed by the two atoms involved. e.g. 1,2,1 is a coordinate for the bond length between atoms 2 and 1.
- 2 indicates a **bond angle** and is followed by the three atoms involved. The central atom is the vertex. e.g. 2,3,1,2 is the bond angle between bonds 3,1 and 1,2.
- 3 indicates a **torsion** and is followed by the four atoms involved. e.g. 3,4,1,2,3 is the angle between the planes defined by atoms 4,1,2 and 1,2,3.

This would **correspond** to the failed *Z-matrix* that we used before we applied the dummy atoms. It is reproduced below.

```text
N
H   1   1.0
H   1   1.0   2   109.5
H   1   1.0   2   109.5   3   120
```

Now *GAMESS* will **track** those coordinates. In fact, *GAMESS* will be using those coordinates to calculate **gradients**. Rather than a gradient in each axis for each atom, *GAMESS* will use the gradient for the energy function of that **internal coordinate**. With **carefully designed** internal coordinates, this can speed the calculation quite a bit. Below is the input file for an AM1 optimization. The book's copy, `inputs/ammonia_UZ.inp`, already contains the unique coordinates from the AM1 job above; we let `gt.new_input()` put in the coordinates from **your** run so that the two jobs are linked.

In [22]:
ammonia_UZ_inp = gt.new_input(
    INPUTS / "ammonia_UZ.inp", RUNS / "ammonia_UZ.inp",
    atoms=gt.final_geometry(ammonia_ZX_log, unique=True),
)
print(ammonia_UZ_inp.read_text())

 $CONTRL SCFTYP=RHF RUNTYP=OPTIMIZE MAXIT=30 MULT=1 ICHARG=0 
  COORD=UNIQUE NZVAR=6 $END
 $SYSTEM TIMLIM=525600 MEMORY=1000000 $END
 $BASIS GBASIS=AM1 $END
 $SCF DIRSCF=.TRUE. $END
 $STATPT OPTTOL=0.0001 NSTEP=100 $END
 $DATA 
Ammonia
CNV 3

 N        7.0    0.0000000000   -0.0000000000   -0.0630644261
 H        1.0    0.9385359335   -0.0000000000    0.2757943561
 $END
 $ZMAT IZMAT(1)=1,2,1,  
                1,3,1,  2,3,1,2, 
                1,4,1,  2,4,1,2,  3,4,1,2,3   $END   



```{tip}
Observe that the `$CONTRL` group is split over two lines in this example. The single line was more than 80 columns and no characters after that will be read into the system. We can have as many new lines as we want. The group is terminated by the `$END` string, not the end of a line.
```

In [23]:
ammonia_UZ_log = gt.run_gamess(ammonia_UZ_inp)
gt.excerpt(ammonia_UZ_log, "NSERCH:", after=30)

ammonia_UZ: finished normally (ammonia_UZ.log)
  664   NSERCH:   0  E=       -9.1354556208  GRAD. MAX=  0.0000009  R.M.S.=  0.0000004
  665  
  666  
  667        ***** EQUILIBRIUM GEOMETRY LOCATED *****
  668   COORDINATES OF SYMMETRY UNIQUE ATOMS (ANGS)
  669     ATOM   CHARGE       X              Y              Z
  670   ------------------------------------------------------------
  671   N           7.0   0.0000000000  -0.0000000000  -0.0630644261
  672   H           1.0   0.9385359335  -0.0000000000   0.2757943561
  673   COORDINATES OF ALL ATOMS ARE (ANGS)
  674     ATOM   CHARGE       X              Y              Z
  675   ------------------------------------------------------------
  676   N           7.0   0.0000000000  -0.0000000000  -0.0630644261
  677   H           1.0  -0.4692679668   0.8127959608   0.2757943561
  678   H           1.0  -0.4692679668  -0.8127959608   0.2757943561
  679   H           1.0   0.9385359335  -0.0000000000   0.2757943561
  680  
  681  
  682   

Take note of the **list** of internal coordinates that was written out by *GAMESS*. Each is **classified** as a STRETCH, BEND or TORSION. This **corresponds** to the 1, 2 and 3 that we used to set up each coordinate. The 4,1,2,3 torsion is not 120&deg; like we had set up earlier. This is why the *Z-matrix* failed when we enforced *C<sub>3v</sub>* symmetry.

Since the `$DATA` group was copied from the result of the **same** AM1 method, we started at a stationary point and so no optimization was performed. What is **different** is that *GAMESS* now includes the list of internal coordinates with the structural data. We can easily **see the values** for all the lengths and angles. `gt.internal_coordinates()` reads that table into *Python*.

In [24]:
for kind, atoms, value in gt.internal_coordinates(ammonia_UZ_log):
    unit = "Å" if kind == "STRETCH" else "°"
    print(f"{kind:8s} {str(atoms):14s} {value:12.4f} {unit}")

STRETCH  (2, 1)               0.9978 Å
STRETCH  (3, 1)               0.9978 Å
BEND     (3, 1, 2)          109.0876 °
STRETCH  (4, 1)               0.9978 Å
BEND     (4, 1, 2)          109.0876 °
TORSION  (4, 1, 2, 3)      -119.0724 °


We observe that the bond lengths are all **exactly** the same and the bond angles are also **exactly** the same. AM1 bond lengths and angles are 1.00 Å and 109.1&deg;.

We can take the result of the AM1 calculation and use it to build an **input file** for HF/3-21G and HF/6-311G(d,p)++ optimizations. Below is a file for the **more expensive** option.

```{literalinclude} inputs/ammonia_UZ_6311G.inp
:label: ammonia_uz_6311g-inp1
:caption: inputs/ammonia_UZ_6311G.inp
:linenos: True
```

In the above input file, I set up the `IZMAT` matrix differently and applied a **bend** rather than a **torsion** for the last coordinate. This way I have coordinates that describe all three bond lengths and all three bends. I can do this because the torsion angles are set by the **symmetry group** (spaced evenly around the principal axis.) Now see how that is reflected in the results. **Be careful** here, *GAMESS* must have a way to define the torsion angle via an internal coordinate when we are using internal coordinates. Here we had the symmetry constraints to handle that role. That won't always be the case.

As before, we put in the AM1 coordinates from our own run and submit the job.

In [25]:
ammonia_6311G_inp = gt.new_input(
    INPUTS / "ammonia_UZ_6311G.inp", RUNS / "ammonia_UZ_6311G.inp",
    atoms=gt.final_geometry(ammonia_UZ_log, unique=True),
)
ammonia_6311G_log = gt.run_gamess(ammonia_6311G_inp)
gt.excerpt(ammonia_6311G_log, "NSERCH:", after=30)

ammonia_UZ_6311G: finished normally (ammonia_UZ_6311G.log)
 1634   NSERCH:   3  E=      -56.2147746439  GRAD. MAX=  0.0000023  R.M.S.=  0.0000013
 1635  
 1636  
 1637        ***** EQUILIBRIUM GEOMETRY LOCATED *****
 1638   COORDINATES OF SYMMETRY UNIQUE ATOMS (ANGS)
 1639     ATOM   CHARGE       X              Y              Z
 1640   ------------------------------------------------------------
 1641   N           7.0  -0.0000000000  -0.0000000000  -0.0652973431
 1642   H           1.0   0.9365806163   0.0000000000   0.2861359970
 1643   COORDINATES OF ALL ATOMS ARE (ANGS)
 1644     ATOM   CHARGE       X              Y              Z
 1645   ------------------------------------------------------------
 1646   N           7.0  -0.0000000000  -0.0000000000  -0.0652973431
 1647   H           1.0  -0.4682903082   0.8111026064   0.2861359970
 1648   H           1.0  -0.4682903082  -0.8111026064   0.2861359970
 1649   H           1.0   0.9365806163   0.0000000000   0.2861359970
 1650  
 165

Now we see the HF/6-311G(d,p)++ results have bond **lengths** of 1.00 Å and bond **angles** of 108.4&deg;.

## Summary

We have explored using *GAMESS* to **calculate** the structure and energy of some small molecules. We tried using **Cartesian coordinates** to describe the initial structure and using **internal coordinates** with the *Z-matrix*. We have used **symmetry** to simplify the structure to the **unique Cartesian coordinates** needed to describe the structure in a **point group**.

We have run *GAMESS* using text input files that we created, submitting the jobs from *Python* with `run_gamess()`. We let *Python* write the input files for the next level of theory, carrying the coordinates forward from one result to the next.

We have used the `$ZMAT` group to **establish internal coordinates** when using Cartesian coordinates so that bond lengths and angles can be easily **interpreted directly** from the text result file. We have examined the **text output**, both by eye and with *Python*, and determined that the results were as expected.

Now go to the instructions for the exercise that this discussion supports and explore *GAMESS* for yourself.

## Exercise

Read the accompanying introduction to *GAMESS*, "Getting Started with Small Molecules", and then try the following exercises.

### The Activity

**Write** an input file for the tetrahedral methane molecule. **Use** the *T<sub>d</sub>* point group with a *Z-matrix* for the first attempt. Look to the ammonia *Z-matrix* with the dummy atom for inspiration. **Run** the job on your computer with `gt.run_gamess()`. If you are successful, you will have accomplished the first four of our learning goals. If it was not successful, **try another way** to build the molecule so that it fits *T<sub>d</sub>* symmetry. Perhaps a different construction of the *Z-matrix* or a set of Cartesian coordinates would work. Errors are learning opportunities. Give yourself time to make lots of them.

After you get it working, copy the coordinate data into **higher** level input files (by hand or with `gt.new_input()`) and **optimize** for HF/3-21G and HF/6-311G(d,p)++ levels of theory.

If you use the optimized *Z-matrix* (no dummy atoms were needed for methane) then you will have a *Z-matrix* in the results from which you can locate **bond lengths and angles**. If you prefer the unique coordinates you can set up a `$ZMAT` group to get those values directly from the output, or measure them with `gt.distance()` and `gt.angle()`.

What are the bond lengths and angles for **methane** at the three levels of theory? Perform HF/6-311G(d,p)++ energy calculations using the optimized coordinates from the results for all three levels of theory. How do they **compare**? Which had the lowest energy (and was therefore the most accurate of the three structures)?

Using the input files in this document **repeat** the above for water and ammonia. **Compare** the bond angles and bond lengths for the most accurate structures in each case.

### The Report

**Create** a ONE PAGE document summarizing your **results** (energies, bond lengths and angles for methane) in a table.  Do the results fit with what you know about VSEPR theory?

In the table, **report** the energies for AM1//HF/6-311G(d,p)++ (optimized at AM1, energy calculated at HF/6-311G(d,p)++), HF/3-21G//HF/6-311G(d,p)++ and HF/6-311G(d,p)++ for each of the three molecules. The lowest energy should be the most accurate. Which was it in each case?

**Comment** on how you did all this in a completely text-based world. Did you miss **visualizing** the molecules in 3-D graphics?  More complicated molecules could benefit from using visual molecule building/display programs. But, when you really want to control symmetry, it is best to manually create the input file.

```{warning}
Remember that the visualization programs work for the machines. Sometimes you must look at the real world in its **raw text** format.
```

Include an appendix with the text for all the input files that you used. (Please do not include any result files – there will be no trees left!) Preface each with the file name and a note on its purpose and use a fixed-width font.

### Example Exercise Report

The example report below was produced by running the whole chain of jobs in *Python*. The function `level_ladder()` takes an AM1 starting input and does what we did by hand for water above:

1. optimize at AM1,
2. copy the AM1 structure into a HF/3-21G input and optimize,
3. copy the HF/3-21G structure into a HF/6-311G(d,p)++ input and optimize,
4. compute single point HF/6-311G(d,p)++ energies at all three structures.

It returns a table of bond lengths, bond angles and energies. Read it before you run it, then try to write your own version for the exercise.

In [26]:
import pandas as pd

BASIS_321 = "GBASIS=N21 NGAUSS=3"


def level_ladder(name, am1_input, coord_change=None):
    # Optimize AM1 -> HF/3-21G -> HF/6-311G(d,p)++, then HF/6-311G(d,p)++
    # single point energies at all three structures.
    # Atom 1 must be the central atom and atoms 2 and 3 hydrogens.
    swap = dict(coord_change or {})              # e.g. {"COORD=ZMT": "COORD=UNIQUE"}
    opt_logs = {"AM1": gt.run_gamess(am1_input)}

    log_321 = opt_logs["AM1"]
    for level, basis, tag in [("HF/3-21G", BASIS_321, "321_U"),
                              ("HF/6-311G(d,p)++", BASIS_6311, "6311_U")]:
        inp = gt.new_input(am1_input, RUNS / f"{name}_{tag}.inp",
                           atoms=gt.final_geometry(log_321, unique=True),
                           replacements={**swap, "GBASIS=AM1": basis})
        log_321 = opt_logs[level] = gt.run_gamess(inp)

    rows = []
    for level, log in opt_logs.items():
        tag = level.split("/")[-1].replace("(d,p)++", "").replace("-", "")
        inp = gt.new_input(am1_input, RUNS / f"{name}_6311_U_E_{tag}.inp",
                           atoms=gt.final_geometry(log, unique=True),
                           replacements={**swap, "GBASIS=AM1": BASIS_6311,
                                         "RUNTYP=OPTIMIZE": "RUNTYP=ENERGY"})
        geom = gt.final_geometry(log)
        rows.append({"Optimization": level,
                     "bond length (Å)": gt.distance(geom, 1, 2),
                     "H–X–H angle (°)": gt.angle(geom, 2, 1, 3),
                     "HF/6-311G(d,p)++ energy (hartree)": gt.final_energy(gt.run_gamess(inp))})
    return pd.DataFrame(rows)


def add_experiment(table, length, angle, energy_log):
    # Add the experimental structure and the energy difference column.
    table.loc[len(table)] = ["Experimental", length, angle, gt.final_energy(energy_log)]
    e_ref = table.loc[2, "HF/6-311G(d,p)++ energy (hartree)"]
    table["ΔE (kJ/mol)"] = (table["HF/6-311G(d,p)++ energy (hartree)"] - e_ref) * HARTREE_TO_KJMOL
    return table.style.format({"bond length (Å)": "{:.7f}", "H–X–H angle (°)": "{:.7f}",
                               "HF/6-311G(d,p)++ energy (hartree)": "{:.10f}",
                               "ΔE (kJ/mol)": "{:+.3f}"}).hide(axis="index")

#### Methane Calculations

The input file was created using **unique** Cartesian coordinates because the *Z-matrix* conversion by *GAMESS* **failed** to achieve *T<sub>d</sub>* symmetry. (Bond angles of 109.5&deg; did not fit the definition of *T<sub>d</sub>*. They should be 109.4712206&deg;.) Once that problem was solved, the **geometry** was calculated at AM1, HF/3-21G and HF/6-311G(d,p)++ levels. Then the energy was calculated at HF/6-311G(d,p)++ for all three structures.

The experimental energy row uses the structure from the [Computational Chemistry Comparison and Benchmark DataBase](https://cccbdb.nist.gov) (C–H 1.087 Å). Even with many decimal places, *GAMESS* still did not recognize the **symmetry** of that *Z-matrix*, so that input uses *C<sub>1</sub>* symmetry instead. Note that there is NO BLANK LINE after the *C<sub>1</sub>* symmetry designation.

##### Methane Data

In [27]:
methane = level_ladder("methane", INPUTS / "methane_AM1_U.inp")
add_experiment(methane, 1.087, 109.471, gt.run_gamess(INPUTS / "methane_6311_Z_E.inp"))

methane_AM1_U: finished normally (methane_AM1_U.log)


methane_321_U: finished normally (methane_321_U.log)


methane_6311_U: finished normally (methane_6311_U.log)


methane_6311_U_E_AM1: finished normally (methane_6311_U_E_AM1.log)


methane_6311_U_E_321G: finished normally (methane_6311_U_E_321G.log)


methane_6311_U_E_6311G: finished normally (methane_6311_U_E_6311G.log)


methane_6311_Z_E: finished normally (methane_6311_Z_E.log)


Optimization,bond length (Å),H–X–H angle (°),"HF/6-311G(d,p)++ energy (hartree)",ΔE (kJ/mol)
AM1,1.1116153,109.4712206,-40.2072444763,+5.053
HF/3-21G,1.0829028,109.4712206,-40.2091641123,+0.013
"HF/6-311G(d,p)++",1.0842731,109.4712206,-40.2091691920,+0.000
Experimental,1.0870000,109.4710000,-40.2091492019,+0.052


##### Commentary

The HF/3-21G **optimization** produced a structure that is **very close** to the HF/6-311G(d,p)++ optimized structure and the **energy** was calculated to be very **similar** as well. The AM1 structure and energy was farther away.

#### Water Calculations

The **first** input file was created with a *Z-matrix* and the geometry was calculated at AM1, HF/3-21G and HF/6-311G(d,p)++ levels. Then the **energy** was calculated at HF/6-311G(d,p)++ for all three structures.

##### Water Data

In [28]:
water = level_ladder("water", INPUTS / "water_AM1_U.inp", {"COORD=ZMT": "COORD=UNIQUE"})
add_experiment(water, 0.958, 104.48, gt.run_gamess(INPUTS / "water_6311_Z_E.inp"))

water_AM1_U: finished normally (water_AM1_U.log)


water_321_U: finished normally (water_321_U.log)


water_6311_U: finished normally (water_6311_U.log)


water_6311_U_E_AM1: finished normally (water_6311_U_E_AM1.log)


water_6311_U_E_321G: finished normally (water_6311_U_E_321G.log)


water_6311_U_E_6311G: finished normally (water_6311_U_E_6311G.log)


water_6311_Z_E: finished normally (water_6311_Z_E.log)


Optimization,bond length (Å),H–X–H angle (°),"HF/6-311G(d,p)++ energy (hartree)",ΔE (kJ/mol)
AM1,0.9613152,103.5345005,-76.0524943643,+2.499
HF/3-21G,0.9666698,107.6785306,-76.0519604402,+3.901
"HF/6-311G(d,p)++",0.9412543,106.2149647,-76.0534461704,+0.000
Experimental,0.9580000,104.4800000,-76.0528152807,+1.656


##### Commentary

The HF/3-21G optimization **produced** a structure that is close to the HF/6-311G(d,p)++ optimized structure and the energy was **calculated** to be similar as well. The AM1 structure and energy seem closer, but not by much.

#### Ammonia Calculations

The first input file was **created** with a *Z-matrix* with a dummy atom and the geometry was calculated at AM1, HF/3-21G and HF/6-311G(d,p)++ levels. Then the **energy** was calculated at HF/6-311G(d,p)++ for all three structures.

##### Ammonia Data

In [29]:
ammonia = level_ladder("ammonia", INPUTS / "ammonia_AM1_ZD.inp", {"COORD=ZMT": "COORD=UNIQUE"})
add_experiment(ammonia, 1.0124, 106.67, gt.run_gamess(INPUTS / "ammonia_6311_Z_E.inp"))

ammonia_AM1_ZD: finished normally (ammonia_AM1_ZD.log)


ammonia_321_U: finished normally (ammonia_321_U.log)


ammonia_6311_U: finished normally (ammonia_6311_U.log)


ammonia_6311_U_E_AM1: finished normally (ammonia_6311_U_E_AM1.log)


ammonia_6311_U_E_321G: finished normally (ammonia_6311_U_E_321G.log)


ammonia_6311_U_E_6311G: finished normally (ammonia_6311_U_E_6311G.log)


ammonia_6311_Z_E: finished normally (ammonia_6311_Z_E.log)


Optimization,bond length (Å),H–X–H angle (°),"HF/6-311G(d,p)++ energy (hartree)",ΔE (kJ/mol)
AM1,0.9978605,109.0887396,-56.2147354804,+0.103
HF/3-21G,1.0025827,112.3967370,-56.2137244548,+2.757
"HF/6-311G(d,p)++",1.0003408,108.3541849,-56.2147746439,+0.000
Experimental,1.0124000,106.6700000,-56.2143301556,+1.167


##### Commentary

The HF/3-21G **optimization** produced a structure that is NOT close to the HF/6-311G(d,p)++ optimized structure. The AM1 structure and energy seems **closer**, but not by much. The HF/6-311G(d,p)++ seems to have given the most **accurate** structure compared to the experimental data.

#### Appendix: Input files

The input files written by hand are in the `inputs/` folder. The others were written by `level_ladder()` into `runs/`. The cell below prints them all, each with its file name.

In [30]:
hand_written = ["methane_AM1_U", "methane_6311_Z_E", "water_AM1_U", "water_6311_Z_E",
                "ammonia_AM1_ZD", "ammonia_6311_Z_E"]
files = [INPUTS / f"{name}.inp" for name in hand_written]
files += sorted(RUNS.glob("methane_*_U*.inp")) + sorted(RUNS.glob("water_*_U*.inp")) \
       + sorted(RUNS.glob("ammonia_*_U*.inp"))
for f in files:
    print(f"===== {f} =====")
    print(f.read_text())

===== inputs/methane_AM1_U.inp =====
 $CONTRL SCFTYP=RHF RUNTYP=OPTIMIZE MAXIT=30 MULT=1 NZVAR=9 $END
 $SYSTEM TIMLIM=525600 MEMORY=1000000 $END
 $BASIS GBASIS=AM1 $END
 $SCF DIRSCF=.TRUE. $END
 $STATPT OPTTOL=0.0001 NSTEP=100 $END
 $DATA 
Methane
Td

C     6.0     0.0    0.0     0.0
H     1.0     0.7    0.7     0.7
 $END
 $ZMAT IZMAT(1)= 1,2,1
                 1,3,1,  2,3,1,2
                 1,4,1,  2,4,1,3,  3,4,1,3,2
                 1,5,1,  2,5,1,4,  3,5,1,4,3
                 $END

===== inputs/methane_6311_Z_E.inp =====
 $CONTRL SCFTYP=RHF RUNTYP=ENERGY MAXIT=30 MULT=1 ICHARG=0 COORD=ZMT $END
 $SYSTEM TIMLIM=525600 MEMORY=1000000 $END
 $BASIS GBASIS=N311 NGAUSS=6 NDFUNC=1 NPFUNC=1 DIFFSP=.TRUE. DIFFS=.TRUE. $END
 $SCF DIRSCF=.TRUE. $END
 $STATPT OPTTOL=0.0001 NSTEP=100 $END
 $DATA 
Methane
C1
H
C  1  1.087
H  2  1.087  1  109.4712206
H  2  1.087  1  109.4712206  3  +120.0
H  2  1.087  1  109.4712206  3  -120.0
 $END

===== inputs/water_AM1_U.inp =====
 $CONTRL SCFTYP=RHF RUNTYP=

## AI Statement

This page was converted from the original Markdown page `2_SmallMolecules.md` to a *Jupyter* notebook with the help of Claude (Anthropic), following the author's instructions.

- **Original text kept (about 85 %).** The explanations of coordinates, symmetry, the input-file groups, levels of theory, units, the *Z-matrix*, dummy atoms, the `$ZMAT` group, the summary, the exercise and the example report commentary are the author's words. Small edits: typos fixed, the water bond-length formula corrected, the `$SCF`, `OPTTOL` and `MEMORY` descriptions made more precise, the wrong point group (*C<sub>2v</sub>* for ammonia) and line-number references fixed, and "cut and paste" / "search the log" sentences reworded for *Python*.
- **Written by AI.** The settings cell, all code cells, the paragraphs that introduce them, the column-80 checker, the Cartesian vs *Z-matrix* energy comparison, the `level_ladder()` function, and the new helpers in `gamess_tools.py` (`excerpt`, `final_geometry`, `distance`, `angle`, `internal_coordinates`, `new_input`). The input `inputs/water_6311_Z_E.inp` (water at its experimental structure) is new.
- **Changed workflow.** Jobs are run with `run_gamess()` instead of `gms`; log excerpts are printed live from your own runs instead of being pasted in; new input files for higher levels of theory are written by *Python* from the previous result; the example report tables are computed rather than typed, so numbers may differ in the last digits from the original page.
- **Comparisons removed.** At the author's request, sentences and shell scripts that compared each step with the previous book's terminal and text-editor workflow were removed or rewritten, so the page reads as written for *Python*.